# Learned Stage-Wise Curvature Tuning on Imagenette

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Imagenette dataset using an ImageNet-pretrained ResNet-50.

Imagenette is a smaller subset of ImageNet containing ten visually distinct classes. Because the backbone is pretrained on ImageNet, this experiment helps evaluate how Baseline, S-CT, and Learned SW-CT behave when the downstream dataset is closely related to the original pretraining domain.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to examine whether a larger training budget improves performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded for each method to compare computational cost.

In [1]:
!nvidia-smi

Sun Oct  4 19:23:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 592, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 592 (delta 166), reused 198 (delta 115), pack-reused 328 (from 1)
Receiving objects: 100% (592/592), 17.47 MiB | 7.66 MiB/s, done.
Resolving deltas: 100% (297/297), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
6ec796c (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated the notebook with outputs


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 93.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 43.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 109.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 9.8 MB/s eta 0:00:00:00:010:01mm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


## Imagenette Dataset

Before running the experiments, the Imagenette data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_imagenette",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

imagenette.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 7576
Validation samples: 1893
Test samples: 3925
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([6, 7, 2, 1, 1, 0, 6, 4, 4, 7])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Imagenette dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 19:49:34.743 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet50_seed42_epochs20.log
2026-10-03 19:49:34.748 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 201MB/s]
2026-10-03 19:50:44.373 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 19:50:44.449 | INFO     | __main__:main:116 - Number of trainable parameters: 20490
2026-10-03 19:50:44.450 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 19:51:27.780 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.495933, Accuracy: 0.00%
2026-10-03 19:51:28.089 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.355636, Accuracy: 69.09%
2026-10-03 19:51:28.404 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 0.7504

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 20:42:53.142 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet50_seed43_epochs20.log
2026-10-03 20:42:53.145 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 20:44:02.625 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 20:44:02.700 | INFO     | __main__:main:116 - Number of trainable parameters: 20490
2026-10-03 20:44:02.701 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 20:44:56.456 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.313626, Accuracy: 21.88%
2026-10-03 20:44:56.872 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.289212, Accuracy: 74.38%
2026-10-03 20:44:57.247 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 0.716967, Accuracy: 86.80%
2026-10-03 20:44:57.649 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.069780, Val Accuracy: 99.52%
2026-10-03 20:44:57.651 | INFO     | train:linear_probe:190 - New best validation a

In [6]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 21:37:38.965 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet50_seed44_epochs20.log
2026-10-03 21:37:38.969 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 21:38:53.489 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 21:38:53.565 | INFO     | __main__:main:116 - Number of trainable parameters: 20490
2026-10-03 21:38:53.566 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 21:39:46.779 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.417753, Accuracy: 0.00%
2026-10-03 21:39:47.190 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.342412, Accuracy: 69.31%
2026-10-03 21:39:47.591 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 0.746661, Accuracy: 84.10%
2026-10-03 21:39:48.003 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.070668, Val Accuracy: 99.21%
2026-10-03 21:39:48.004 | INFO     | train:linear_probe:190 - New best validation ac

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 22:31:55.116 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 22:31:55.231 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 22:33:05.524 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 22:33:05.524 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 22:33:05.631 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 22:33:05.633 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 22:33:05.633 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-03 22:33:07.540 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.345512, Accuracy: 15.62%
2026-10-03 22:34:10.544 | INFO     | train:train_epo

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 23:28:00.813 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 23:28:00.913 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 23:29:11.224 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 23:29:11.225 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 23:29:11.325 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 23:29:11.326 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 23:29:11.326 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-03 23:29:13.169 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.343932, Accuracy: 6.25%
2026-10-03 23:30:15.966 | INFO     | train:train_epoc

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 00:24:03.689 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-04 00:24:03.791 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 00:24:57.758 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 00:24:57.758 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 00:24:57.832 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 00:24:57.833 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 00:24:57.833 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 00:24:59.463 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.434211, Accuracy: 6.25%
2026-10-04 00:26:02.047 | INFO     | train:train_epoc

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 01:19:23.764 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-04 01:19:23.878 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 01:20:31.950 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 01:20:31.951 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 01:20:32.049 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-04 01:20:32.051 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 01:20:32.051 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 01:20:33.767 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.317595, Accuracy: 18.75%
2026-10-04 01:21:36.700 | INFO     | tra

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 02:15:27.438 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-04 02:15:27.543 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 02:16:36.511 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 02:16:36.512 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 02:16:36.617 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-04 02:16:36.618 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 02:16:36.618 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 02:16:38.373 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.338269, Accuracy: 12.50%
2026-10-04 02:17:41.101 | INFO     | t

In [12]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 03:11:22.714 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-04 03:11:22.799 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 03:12:16.714 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 03:12:16.714 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 03:12:16.779 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-04 03:12:16.780 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 03:12:16.780 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 03:12:18.361 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.431987, Accuracy: 6.25%
2026-10-04 03:13:21.111 | INFO     | tr

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [13]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 04:07:03.643 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet50_seed42_epochs30.log
2026-10-04 04:07:03.647 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 04:08:14.050 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 04:08:14.102 | INFO     | __main__:main:116 - Number of trainable parameters: 20490
2026-10-04 04:08:14.102 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 04:09:03.943 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.495933, Accuracy: 0.00%
2026-10-04 04:09:04.338 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.355636, Accuracy: 69.09%
2026-10-04 04:09:04.694 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 0.750412, Accuracy: 84.03%
2026-10-04 04:09:05.149 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.069374, Val Accuracy: 99.10%
2026-10-04 04:09:05.151 | INFO     | train:linear_probe:190 - New best validation ac

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 11:18:52.446 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet50_seed43_epochs30.log
2026-10-04 11:18:52.452 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 129MB/s]
2026-10-04 11:19:44.331 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 11:19:44.408 | INFO     | __main__:main:116 - Number of trainable parameters: 20490
2026-10-04 11:19:44.408 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 11:20:36.103 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.324131, Accuracy: 18.75%
2026-10-04 11:20:36.555 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.285619, Accuracy: 74.26%
2026-10-04 11:20:36.940 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 0.716302, Accuracy: 86.55%
2026-10-04 11:20:37

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 12:12:15.178 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet50_seed44_epochs30.log
2026-10-04 12:12:15.181 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 12:13:05.127 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 12:13:05.165 | INFO     | __main__:main:116 - Number of trainable parameters: 20490
2026-10-04 12:13:05.166 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 12:13:57.488 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.437371, Accuracy: 0.00%
2026-10-04 12:13:57.808 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.340977, Accuracy: 69.15%
2026-10-04 12:13:58.079 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 0.746901, Accuracy: 84.05%
2026-10-04 12:13:58.336 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.062776, Val Accuracy: 99.31%
2026-10-04 12:13:58.337 | INFO     | train:linear_probe:190 - New best validation ac

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 13:05:29.003 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 13:05:29.098 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 13:06:17.683 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 13:06:17.683 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 13:06:17.733 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 13:06:17.734 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 13:06:17.735 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 13:06:19.394 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.348498, Accuracy: 12.50%
2026-10-04 13:07:20.005 | INFO     | train:train_epo

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 14:25:11.046 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 14:25:11.113 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 14:25:59.987 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 14:25:59.988 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 14:26:00.040 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 14:26:00.041 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 14:26:00.041 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 14:26:01.529 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.376670, Accuracy: 15.62%
2026-10-04 14:27:02.179 | INFO     | train:train_epo

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 15:44:46.721 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 15:44:46.783 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 15:45:37.512 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 15:45:37.513 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 15:45:37.567 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 15:45:37.568 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 15:45:37.568 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 15:45:39.028 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.426271, Accuracy: 6.25%
2026-10-04 15:46:39.699 | INFO     | train:train_epoc

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 17:04:26.599 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-04 17:04:26.662 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 17:05:17.239 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 17:05:17.240 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 17:05:17.291 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-04 17:05:17.292 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 17:05:17.292 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 17:05:18.840 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.317623, Accuracy: 18.75%
2026-10-04 17:06:19.589 | INFO     | tra

In [3]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 19:31:58.878 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-04 19:31:58.948 | INFO     | __main__:main:302 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 175MB/s]
2026-10-04 19:32:50.387 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 19:32:50.388 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 19:32:50.488 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-04 19:32:50.489 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 19:32:50.489 | INFO     | __main__:main:439 - Total tra

In [4]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 20:52:37.971 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-04 20:52:38.060 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 20:53:27.531 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 20:53:27.531 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 20:53:27.617 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-04 20:53:27.618 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 20:53:27.618 | INFO     | __main__:main:439 - Total trainable parameters: 20494
2026-10-04 20:53:29.809 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.392361, Accuracy: 12.50%
2026-10-04 20:54:28.573 | INFO     | t